# S4 · AndinaLog 03B · 1/7 · Diagnóstico de telemetría IoT

Este primer notebook **solo diagnostica** el CSV Bronze. Conserva los valores originales, marca `en_cuarentena` y registra todos los `motivos_cuarentena` de cada fila. No convierte unidades, corrige valores ni imputa faltantes. La curación se hará en un segundo notebook, después de acordar las reglas.

El archivo principal `andinalog_iot_telemetry_diagnosticado.csv` contiene **todas** las filas. El archivo `andinalog_iot_telemetry_cuarentena.csv` es un extracto informativo; el reporte resume los controles. Los tres se guardan en `S4/salidas/` y nunca sobrescriben `datasets/`.


In [13]:
from pathlib import Path
from datetime import datetime
import hashlib
import re
import unicodedata
import pandas as pd

# Local: ejecuta desde cualquier carpeta dentro del proyecto.
# Colab: monta Drive y configura RUTA_PROYECTO_DRIVE con tu carpeta real.
ENTORNO = "drive"  # "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"

def encontrar_raiz_local():
    for candidata in [Path.cwd(), *Path.cwd().parents]:
        if (candidata / "datasets" / "GIAD_M3_Subcaso_03B_Bronce").is_dir() and (candidata / "S4").is_dir():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del proyecto. Ejecuta dentro de practicasNotebookColab.")

if ENTORNO == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_PROYECTO = Path(RUTA_PROYECTO_DRIVE)
else:
    RAIZ_PROYECTO = encontrar_raiz_local()

RUTA_BRONZE = RAIZ_PROYECTO / "datasets" / "AndinaLog_03B_Bronce" / "andinalog_iot_telemetry.csv"
DIRECTORIO_SALIDAS = RAIZ_PROYECTO / "S4" / "salidas"
if not RUTA_BRONZE.is_file():
    raise FileNotFoundError(f"No se encontró el CSV Bronze: {RUTA_BRONZE}")
HASH_BRONZE = hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
print("Origen:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)
print("SHA-256 Bronze:", HASH_BRONZE)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Origen: /content/drive/MyDrive/GIAD/datasets/AndinaLog_03B_Bronce/andinalog_iot_telemetry.csv
Salidas: /content/drive/MyDrive/GIAD/S4/salidas
SHA-256 Bronze: c5f78ed802f8455dfc8eac91295954c38a159029e77b8ed748875fe2d8cf7467


## Reglas de detección de este CSV

Se comprueba el esquema, los identificadores vacíos, el formato y validez del timestamp, la duplicación de la clave `viaje_id + timestamp`, temperatura y humedad faltantes o no numéricas, humedad fuera del rango físico 0–100%, banderas distintas de 0/1 y unidad de temperatura distinta de C/F. La unidad K se marca para revisión porque aún no hay una regla de tratamiento acordada. El timestamp se verifica como fecha, pero **no se convierte ni se incorpora una fecha transformada** en la salida.

La clave y estos controles son reglas iniciales de diagnóstico; pueden revisarse con el responsable del dato antes de curar. Una fila puede acumular varios motivos. El valor original permanece intacto, incluso si está en cuarentena.


In [14]:
COLUMNAS_REQUERIDAS = [
    "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
    "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
    "desviacion_termica_flag", "desviacion_proximos_60min_flag",
]
VERSION_DIAGNOSTICO = "GIAD-M3-S4-IOT-diagnostico-v1"

df_bronze = pd.read_csv(RUTA_BRONZE, dtype="string", encoding="utf-8-sig", keep_default_na=False)
snapshot_bronze = df_bronze.copy(deep=True)

def snake(nombre):
    base = unicodedata.normalize("NFKD", str(nombre))
    base = "".join(c for c in base if not unicodedata.combining(c))
    return re.sub(r"_+", "_", re.sub(r"[^a-z0-9]+", "_", base.lower())).strip("_")

nombres_normalizados = [snake(c) for c in df_bronze.columns]
if len(set(nombres_normalizados)) != len(nombres_normalizados):
    raise ValueError("Hay nombres de columna duplicados tras normalizarlos")
faltantes = sorted(set(COLUMNAS_REQUERIDAS) - set(nombres_normalizados))
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {faltantes}")
if nombres_normalizados != list(df_bronze.columns):
    raise ValueError("El esquema requiere renombrar columnas; revisar antes de alterar el Bronze")
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
display(df_bronze.head())


Bronze: 28,920 filas × 10 columnas


,timestamp,viaje_id,order_id,camion_id,producto_id,temperatura_cabina_c,temp_unit,humedad_cabina_pct,desviacion_termica_flag,desviacion_proximos_60min_flag
0,2026-08-04 20:15:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-18.75,C,74.2,0,0
1,2026-08-04 20:45:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-18.33,C,65.0,0,0
2,2026-08-04 21:15:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-17.52,C,73.0,0,0
3,2026-08-04 21:45:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-17.96,C,85.4,0,0
4,2026-08-04 22:15:00,VIA-00001,ORD-2026-05710,CAM-12,PROD-052,-18.96,C,60.6,0,0


In [15]:
def marcar_cuarentena(df_origen):
    df = df_origen.copy(deep=True)
    df.insert(0, "fila_bronze", range(1, len(df) + 1))

    def texto(col):
        return df[col].astype("string").str.strip()

    def vacio(col):
        return texto(col).eq("").fillna(True)

    timestamp = texto("timestamp")
    fecha_valida = timestamp.map(lambda x: bool(re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", x)) and _fecha_real(x))

    def numerico(col):
        valor = texto(col)
        numero = pd.to_numeric(valor, errors="coerce")
        return valor, numero

    temp_txt, temp_num = numerico("temperatura_cabina_c")
    hum_txt, hum_num = numerico("humedad_cabina_pct")
    termica_txt, termica_num = numerico("desviacion_termica_flag")
    prox_txt, prox_num = numerico("desviacion_proximos_60min_flag")

    clave = pd.DataFrame({"viaje": texto("viaje_id"), "timestamp": timestamp})
    banderas = {
        "timestamp_invalido": ~fecha_valida,
        "viaje_id_faltante": vacio("viaje_id"),
        "order_id_faltante": vacio("order_id"),
        "camion_id_faltante": vacio("camion_id"),
        "producto_id_faltante": vacio("producto_id"),
        "lectura_duplicada": clave.duplicated(keep="first"),
        "unidad_no_reconocida": ~texto("temp_unit").str.upper().isin(["C", "F"]),
        "temperatura_faltante": temp_txt.eq(""),
        "temperatura_no_numerica": temp_txt.ne("") & temp_num.isna(),
        "humedad_faltante": hum_txt.eq(""),
        "humedad_no_numerica": hum_txt.ne("") & hum_num.isna(),
        "humedad_fuera_rango": hum_num.notna() & ~hum_num.between(0, 100),
        "flag_termica_invalida": ~termica_num.isin([0, 1]) | termica_txt.eq(""),
        "flag_60min_invalida": ~prox_num.isin([0, 1]) | prox_txt.eq(""),
    }
    nombres = list(banderas)
    for nombre, valores in banderas.items():
        df[nombre] = valores.fillna(False).astype(bool)
    df["motivos_cuarentena"] = df[nombres].apply(
        lambda fila: "|".join(nombre for nombre in nombres if fila[nombre]), axis=1
    )
    df["en_cuarentena"] = df["motivos_cuarentena"].ne("")
    df["version_diagnostico"] = VERSION_DIAGNOSTICO
    return df, nombres

def _fecha_real(valor):
    try:
        datetime.strptime(valor, "%Y-%m-%d %H:%M:%S")
        return True
    except (ValueError, TypeError):
        return False

df_diagnosticado, nombres_banderas = marcar_cuarentena(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
pd.testing.assert_frame_equal(df_bronze, snapshot_bronze)
pd.testing.assert_frame_equal(df_diagnosticado[COLUMNAS_REQUERIDAS], df_bronze[COLUMNAS_REQUERIDAS])
assert len(df_diagnosticado) == len(df_bronze)
assert len(df_cuarentena) == int(df_diagnosticado["en_cuarentena"].sum())
assert df_diagnosticado["fila_bronze"].is_unique
print(f"Diagnóstico: {len(df_diagnosticado):,} filas; en cuarentena: {len(df_cuarentena):,}")
display(df_cuarentena["motivos_cuarentena"].value_counts().head(15))


Diagnóstico: 28,920 filas; en cuarentena: 330


,count
motivos_cuarentena,
lectura_duplicada,116
humedad_faltante,97
temperatura_faltante,78
timestamp_invalido,15
humedad_fuera_rango,15
unidad_no_reconocida,4
lectura_duplicada|temperatura_faltante,2
lectura_duplicada|humedad_faltante,2
unidad_no_reconocida|humedad_faltante,1


In [16]:
reporte = pd.DataFrame({
    "metrica": ["filas_bronze", "filas_diagnosticadas", "filas_en_cuarentena", "filas_sin_cuarentena"] + nombres_banderas,
    "valor": [len(df_bronze), len(df_diagnosticado), len(df_cuarentena), len(df_diagnosticado) - len(df_cuarentena)]
             + [int(df_diagnosticado[c].sum()) for c in nombres_banderas],
})
display(reporte)


,metrica,valor
0,filas_bronze,28920
1,filas_diagnosticadas,28920
2,filas_en_cuarentena,330
3,filas_sin_cuarentena,28590
4,timestamp_invalido,15
5,viaje_id_faltante,0
6,order_id_faltante,0
7,camion_id_faltante,0
8,producto_id_faltante,0
9,lectura_duplicada,120


In [17]:
DIRECTORIO_SALIDAS.mkdir(parents=True, exist_ok=True)
rutas = {
    "principal": DIRECTORIO_SALIDAS / "andinalog_iot_telemetry_diagnosticado.csv",
    "cuarentena": DIRECTORIO_SALIDAS / "andinalog_iot_telemetry_cuarentena.csv",
    "reporte": DIRECTORIO_SALIDAS / "andinalog_iot_telemetry_reporte_calidad.csv",
}
df_diagnosticado.to_csv(rutas["principal"], index=False, encoding="utf-8-sig")
df_cuarentena.to_csv(rutas["cuarentena"], index=False, encoding="utf-8-sig")
reporte.to_csv(rutas["reporte"], index=False, encoding="utf-8-sig")
assert hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest() == HASH_BRONZE
for nombre, ruta in rutas.items():
    print(nombre, ruta)
print("Bronze intacta; todas las filas están en el archivo principal.")


principal /content/drive/MyDrive/GIAD/S4/salidas/andinalog_iot_telemetry_diagnosticado.csv
cuarentena /content/drive/MyDrive/GIAD/S4/salidas/andinalog_iot_telemetry_cuarentena.csv
reporte /content/drive/MyDrive/GIAD/S4/salidas/andinalog_iot_telemetry_reporte_calidad.csv
Bronze intacta; todas las filas están en el archivo principal.


## Siguiente etapa

El segundo notebook leerá `S4/salidas/andinalog_iot_telemetry_diagnosticado.csv`. Una vez acordadas las reglas de tratamiento, podrá convertir o corregir casos recuperables y volverá a evaluar la cuarentena. Las filas sin solución autorizada permanecerán en cuarentena, con trazabilidad del motivo inicial y del resultado final.
